In [5]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [6]:
import os

DATASET_ZIP = "/content/drive/MyDrive/NewsGuard/liar_dataset.zip"

print("ZIP exists:", os.path.exists(DATASET_ZIP))
print("ZIP path:", DATASET_ZIP)

ZIP exists: True
ZIP path: /content/drive/MyDrive/NewsGuard/liar_dataset.zip


In [7]:
import zipfile
import os

EXTRACT_DIR = "/content/liar_dataset"

with zipfile.ZipFile(DATASET_ZIP, "r") as zip_ref:
    zip_ref.extractall(EXTRACT_DIR)

print("Dataset extracted successfully.")
print("\nFiles:")
for root, dirs, files in os.walk(EXTRACT_DIR):
    for file in files:
        print(os.path.join(root, file))

Dataset extracted successfully.

Files:
/content/liar_dataset/test.tsv
/content/liar_dataset/valid.tsv
/content/liar_dataset/train.tsv
/content/liar_dataset/README


In [8]:
import pandas as pd
import os

DATA_DIR = "/content/liar_dataset"

columns = [
    "label",
    "statement",
    "subject",
    "speaker",
    "speaker_job_title",
    "state_info",
    "party_affiliation",
    "barely_true_counts",
    "false_counts",
    "half_true_counts",
    "mostly_true_counts",
    "pants_on_fire_counts",
    "context"
]

train_raw = pd.read_csv(
    os.path.join(DATA_DIR, "train.tsv"),
    sep="\t",
    header=None,
    names=columns
)

valid_raw = pd.read_csv(
    os.path.join(DATA_DIR, "valid.tsv"),
    sep="\t",
    header=None,
    names=columns
)

test_raw = pd.read_csv(
    os.path.join(DATA_DIR, "test.tsv"),
    sep="\t",
    header=None,
    names=columns
)

print("Train:", train_raw.shape)
print("Validation:", valid_raw.shape)
print("Test:", test_raw.shape)

Train: (10240, 13)
Validation: (1284, 13)
Test: (1267, 13)


In [9]:
# Combine all official LIAR splits
full_df = pd.concat(
    [train_raw, valid_raw, test_raw],
    ignore_index=True
)

# Binary label mapping
fake_labels = ["pants-fire", "false", "barely-true", "half-true"]
real_labels = ["mostly-true", "true"]

full_df["binary_label"] = full_df["label"].apply(
    lambda x: "Fake" if x in fake_labels
    else "Real" if x in real_labels
    else None
)

# Keep only valid binary labels
full_df = full_df.dropna(subset=["binary_label"])

print("Total records:", len(full_df))
print("\nBinary label distribution:")
print(full_df["binary_label"].value_counts())

print("\nPercent distribution:")
print(full_df["binary_label"].value_counts(normalize=True) * 100)

Total records: 12791

Binary label distribution:
binary_label
Fake    8284
Real    4507
Name: count, dtype: int64

Percent distribution:
binary_label
Fake    64.764287
Real    35.235713
Name: proportion, dtype: float64


In [10]:
# Check duplicates before removal
duplicate_count = full_df["statement"].duplicated().sum()

print("Duplicate statements before removal:", duplicate_count)

# Remove duplicate statements
full_df = full_df.drop_duplicates(
    subset=["statement"],
    keep="first"
).reset_index(drop=True)

print("Total records after duplicate removal:", len(full_df))
print("\nDuplicate statements after removal:",
      full_df["statement"].duplicated().sum())

print("\nBinary label distribution:")
print(full_df["binary_label"].value_counts())

Duplicate statements before removal: 26
Total records after duplicate removal: 12765

Duplicate statements after removal: 0

Binary label distribution:
binary_label
Fake    8263
Real    4502
Name: count, dtype: int64


In [11]:
!pip install -q nltk spacy

In [12]:
import nltk

nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download("stopwords")

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

In [13]:
!pip install -q spacy
!python -m spacy download en_core_web_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 66.2 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [14]:
import spacy

nlp = spacy.load("en_core_web_sm")

print("SpaCy version:", spacy.__version__)
print("Model loaded successfully!")

SpaCy version: 3.8.16
Model loaded successfully!


In [15]:
import re
import string
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

stop_words = set(stopwords.words("english"))

def preprocess_text(text):
    # 1. Handle missing values
    if pd.isna(text):
        return ""

    # 2. Convert to lowercase
    text = text.lower()

    # 3. Remove URLs
    text = re.sub(r"http\S+|www\S+|https\S+", "", text)

    # 4. Remove punctuation
    text = text.translate(str.maketrans("", "", string.punctuation))

    # 5. Tokenization
    tokens = word_tokenize(text)

    # 6. Remove stopwords
    tokens = [
        word for word in tokens
        if word not in stop_words
    ]

    # 7. Lemmatization using SpaCy
    doc = nlp(" ".join(tokens))

    lemmas = [
        token.lemma_
        for token in doc
        if not token.is_space
    ]

    return " ".join(lemmas)


print("Preprocessing function created successfully!")

Preprocessing function created successfully!


In [16]:
for i in range(5):
    original = full_df.loc[i, "statement"]
    cleaned = preprocess_text(original)

    print(f"\nExample {i+1}")
    print("Original :", original)
    print("Cleaned  :", cleaned)
    print("-" * 80)


Example 1
Original : Says the Annies List political group supports third-trimester abortions on demand.
Cleaned  : say annie list political group support thirdtrimester abortion demand
--------------------------------------------------------------------------------

Example 2
Original : When did the decline of coal start? It started when natural gas took off that started to begin in (President George W.) Bushs administration.
Cleaned  : decline coal start start natural gas took start begin president george w bushs administration
--------------------------------------------------------------------------------

Example 3
Original : Hillary Clinton agrees with John McCain "by voting to give George Bush the benefit of the doubt on Iran."
Cleaned  : hillary clinton agree john mccain voting give george bush benefit doubt iran
--------------------------------------------------------------------------------

Example 4
Original : Health care reform legislation is likely to mandate free sex cha

In [17]:
from tqdm.auto import tqdm

tqdm.pandas()

full_df["cleaned_statement"] = full_df["statement"].progress_apply(
    preprocess_text
)

print("Preprocessing completed!")
print("Total records:", len(full_df))

print("\nSample:")
print(full_df[["statement", "cleaned_statement", "binary_label"]].head())

  0%|          | 0/12765 [00:00<?, ?it/s]

Preprocessing completed!
Total records: 12765

Sample:
                                           statement  \
0  Says the Annies List political group supports ...   
1  When did the decline of coal start? It started...   
2  Hillary Clinton agrees with John McCain "by vo...   
3  Health care reform legislation is likely to ma...   
4  The economic turnaround started at the end of ...   

                                   cleaned_statement binary_label  
0  say annie list political group support thirdtr...         Fake  
1  decline coal start start natural gas took star...         Fake  
2  hillary clinton agree john mccain voting give ...         Real  
3  health care reform legislation likely mandate ...         Fake  
4                 economic turnaround start end term         Fake  


In [18]:
import os

PROCESSED_DIR = "/content/drive/MyDrive/NewsGuard/processed"

os.makedirs(PROCESSED_DIR, exist_ok=True)

output_path = os.path.join(
    PROCESSED_DIR,
    "liar_preprocessed.csv"
)

full_df.to_csv(output_path, index=False)

print("Preprocessed dataset saved successfully!")
print("Path:", output_path)
print("Shape:", full_df.shape)

Preprocessed dataset saved successfully!
Path: /content/drive/MyDrive/NewsGuard/processed/liar_preprocessed.csv
Shape: (12765, 15)


In [19]:
# Check for empty cleaned statements
empty_cleaned = (full_df["cleaned_statement"].str.strip() == "").sum()

# Check for missing values
missing_cleaned = full_df["cleaned_statement"].isna().sum()

# Check original vs cleaned text length
full_df["original_words"] = full_df["statement"].str.split().str.len()
full_df["cleaned_words"] = full_df["cleaned_statement"].str.split().str.len()

print("Empty cleaned statements:", empty_cleaned)
print("Missing cleaned statements:", missing_cleaned)

print("\nAverage words:")
print("Original:", round(full_df["original_words"].mean(), 2))
print("Cleaned :", round(full_df["cleaned_words"].mean(), 2))

print("\nMinimum cleaned words:", full_df["cleaned_words"].min())
print("Maximum cleaned words:", full_df["cleaned_words"].max())

Empty cleaned statements: 0
Missing cleaned statements: 0

Average words:
Original: 18.06
Cleaned : 11.25

Minimum cleaned words: 1
Maximum cleaned words: 344


In [20]:
from sklearn.model_selection import train_test_split

# First split: 80% train, 20% temporary
train_data, temp_data = train_test_split(
    full_df,
    test_size=0.20,
    stratify=full_df["binary_label"],
    random_state=42
)

# Second split: 10% validation, 10% test
valid_data, test_data = train_test_split(
    temp_data,
    test_size=0.50,
    stratify=temp_data["binary_label"],
    random_state=42
)

print("Train:", train_data.shape)
print("Validation:", valid_data.shape)
print("Test:", test_data.shape)

print("\nClass distribution:")
print("\nTrain:")
print(train_data["binary_label"].value_counts())

print("\nValidation:")
print(valid_data["binary_label"].value_counts())

print("\nTest:")
print(test_data["binary_label"].value_counts())

Train: (10212, 17)
Validation: (1276, 17)
Test: (1277, 17)

Class distribution:

Train:
binary_label
Fake    6610
Real    3602
Name: count, dtype: int64

Validation:
binary_label
Fake    826
Real    450
Name: count, dtype: int64

Test:
binary_label
Fake    827
Real    450
Name: count, dtype: int64


In [21]:
import os

FINAL_DIR = "/content/drive/MyDrive/NewsGuard/processed"

os.makedirs(FINAL_DIR, exist_ok=True)

train_data.to_csv(
    os.path.join(FINAL_DIR, "train_preprocessed.csv"),
    index=False
)

valid_data.to_csv(
    os.path.join(FINAL_DIR, "validation_preprocessed.csv"),
    index=False
)

test_data.to_csv(
    os.path.join(FINAL_DIR, "test_preprocessed.csv"),
    index=False
)

print("Final preprocessed splits saved successfully!")

print("\nFiles:")
print("1. train_preprocessed.csv")
print("2. validation_preprocessed.csv")
print("3. test_preprocessed.csv")

Final preprocessed splits saved successfully!

Files:
1. train_preprocessed.csv
2. validation_preprocessed.csv
3. test_preprocessed.csv


In [22]:
# Check statement overlap between splits
train_statements = set(train_data["statement"])
valid_statements = set(valid_data["statement"])
test_statements = set(test_data["statement"])

train_valid_overlap = train_statements & valid_statements
train_test_overlap = train_statements & test_statements
valid_test_overlap = valid_statements & test_statements

print("===== DATA LEAKAGE CHECK =====")

print("Train ∩ Validation:", len(train_valid_overlap))
print("Train ∩ Test:", len(train_test_overlap))
print("Validation ∩ Test:", len(valid_test_overlap))

print("\n===== DATA QUALITY CHECK =====")

print("Train missing cleaned text:", train_data["cleaned_statement"].isna().sum())
print("Validation missing cleaned text:", valid_data["cleaned_statement"].isna().sum())
print("Test missing cleaned text:", test_data["cleaned_statement"].isna().sum())

print("\nEmpty cleaned statements:")
print("Train:", (train_data["cleaned_statement"].str.strip() == "").sum())
print("Validation:", (valid_data["cleaned_statement"].str.strip() == "").sum())
print("Test:", (test_data["cleaned_statement"].str.strip() == "").sum())

===== DATA LEAKAGE CHECK =====
Train ∩ Validation: 0
Train ∩ Test: 0
Validation ∩ Test: 0

===== DATA QUALITY CHECK =====
Train missing cleaned text: 0
Validation missing cleaned text: 0
Test missing cleaned text: 0

Empty cleaned statements:
Train: 0
Validation: 0
Test: 0
